# Pregel Tutorial: Iterative Graph Algorithms with GraphFrames

This notebook covers GraphFrames' Pregel API for developing scalable, iterative graph algorithms using **Apache Spark 4.0**. We build progressively complex algorithms — from simple degree counting to custom reputation propagation — using real Stack Exchange data.

**Prerequisites**: Complete the [Data Setup Tutorial](https://graphframes.io/03-tutorials/03-data-setup.html) to download and prepare the Stack Exchange dataset before running this notebook.

[Pregel](https://15799.courses.cs.cmu.edu/fall2013/static/papers/p135-malewicz.pdf) is a vertex-centric programming model for distributed graph processing introduced by Google in 2010. The core idea: **think like a vertex** — write a function that executes at each vertex independently, communicating via messages along edges.

## Setup and Data Loading

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql import DataFrame, SparkSession, Window
from graphframes import GraphFrame
from graphframes.lib import AggregateMessages as AM
from graphframes.lib import Pregel

spark: SparkSession = (
    SparkSession.builder.appName("Pregel Tutorial")
    .config("spark.sql.caseSensitive", True)
    .getOrCreate()
)
spark.sparkContext.setCheckpointDir("/tmp/graphframes-checkpoints/pregel")
spark.sparkContext.setLogLevel("WARN")

In [ ]:
# Load Stack Exchange data
STACKEXCHANGE_SITE = "stats.meta.stackexchange.com"
BASE_PATH = f"python/graphframes/tutorials/data/{STACKEXCHANGE_SITE}"

nodes_df: DataFrame = spark.read.parquet(f"{BASE_PATH}/Nodes.parquet")
nodes_df = nodes_df.repartition(50).checkpoint().cache()

edges_df: DataFrame = spark.read.parquet(f"{BASE_PATH}/Edges.parquet")
edges_df = edges_df.repartition(50).checkpoint().cache()

g = GraphFrame(nodes_df, edges_df)
print(f"Nodes: {nodes_df.count():,}")
print(f"Edges: {edges_df.count():,}")

## Example 1: In-Degree with AggregateMessages

AggregateMessages performs a **single pass** of message sending and aggregation. Each source sends 1 to its destination; we sum at each vertex to get in-degree.

In [ ]:
# Each source sends 1 to destination, count at each vertex
am_in_degrees = g.aggregateMessages(
    F.count(AM.msg).alias("in_degree"),
    sendToDst=F.lit(1),
)

# Left join to include zero-degree nodes
complete_in_deg = (
    g.vertices.select("id", "Type")
    .join(am_in_degrees, on="id", how="left")
    .na.fill(0, ["in_degree"])
)

print("In-degree distribution:")
complete_in_deg.groupBy("in_degree").count().orderBy("in_degree").show(10)

print("Top 10 nodes by in-degree:")
complete_in_deg.orderBy(F.desc("in_degree")).show(10)

## Example 2: In-Degree with Pregel

The same algorithm using Pregel. Notice how Pregel automatically handles zero-degree vertices via the initial value.

In [ ]:
pregel_in_degree = (
    g.pregel.setMaxIter(1)
    .withVertexColumn(
        "in_degree",
        F.lit(0),                              # Initial value
        F.coalesce(Pregel.msg(), F.lit(0)),    # Update: use message or keep 0
    )
    .sendMsgToDst(F.lit(1))                    # Send 1 to each destination
    .aggMsgs(F.sum(Pregel.msg()))              # Sum all received messages
    .run()
)

print("In-degree distribution (Pregel):")
pregel_in_degree.select("in_degree").groupBy("in_degree").count().orderBy("in_degree").show(10)

print("Top 10 nodes by in-degree (Pregel):")
pregel_in_degree.select("id", "Type", "in_degree").orderBy(F.desc("in_degree")).show(10)

## Example 3: PageRank with Pregel

PageRank computes vertex importance: a node is important if important nodes point to it. This requires **multiple iterations** where vertex states evolve — a natural fit for Pregel.

Formula: `PR(v) = (1 - d) / N + d × Σ(PR(u) / out_degree(u))`

In [ ]:
# Compute out-degrees
out_degrees = g.outDegrees.withColumnRenamed("outDegree", "out_degree")
pr_vertices = nodes_df.join(out_degrees, on="id", how="left").na.fill(1, ["out_degree"])
g_pr = GraphFrame(pr_vertices, edges_df)

num_vertices = g_pr.vertices.count()
damping = 0.85
max_iter = 10

print(f"Running PageRank: {num_vertices:,} vertices, damping={damping}, max_iter={max_iter}")

In [ ]:
pr_results = (
    g_pr.pregel.setMaxIter(max_iter)
    .withVertexColumn(
        "pagerank",
        F.lit(1.0 / num_vertices),
        F.coalesce(Pregel.msg(), F.lit(0.0)) * F.lit(damping)
        + F.lit((1.0 - damping) / num_vertices),
    )
    .sendMsgToDst(Pregel.src("pagerank") / Pregel.src("out_degree"))
    .aggMsgs(F.sum(Pregel.msg()))
    .run()
)

print("Top 10 Questions by PageRank:")
(
    pr_results.filter(F.col("Type") == "Question")
    .select("id", "Title", "pagerank")
    .orderBy(F.desc("pagerank"))
    .show(10, truncate=50)
)

In [ ]:
# Compare with built-in PageRank
builtin_pr = g.pageRank(resetProbability=1 - damping, maxIter=max_iter)

pregel_ranked = (
    pr_results.select("id", F.col("pagerank").alias("pregel_pr"))
    .withColumn("pregel_rank", F.dense_rank().over(Window.orderBy(F.desc("pregel_pr"))))
)
builtin_ranked = (
    builtin_pr.vertices.select("id", F.col("pagerank").alias("builtin_pr"))
    .withColumn("builtin_rank", F.dense_rank().over(Window.orderBy(F.desc("builtin_pr"))))
)
comparison = pregel_ranked.join(builtin_ranked, on="id")
rank_corr = comparison.stat.corr("pregel_rank", "builtin_rank")
print(f"Rank correlation with built-in PageRank: {rank_corr:.4f}")

builtin_pr.vertices.unpersist()

## Example 4: Connected Components with Pregel

Each vertex starts with its own ID as label. Vertices exchange labels bidirectionally, adopting the **minimum**. After convergence, all vertices in the same component share the same label.

In [ ]:
cc_vertices = g.vertices.select("id")
cc_graph = GraphFrame(cc_vertices, g.edges.select("src", "dst"))

cc_results = (
    cc_graph.pregel.setMaxIter(20)
    .setEarlyStopping(True)
    .withVertexColumn(
        "component",
        F.col("id"),
        F.least(F.col("component"), F.coalesce(Pregel.msg(), F.col("component"))),
    )
    .sendMsgToDst(Pregel.src("component"))
    .sendMsgToSrc(Pregel.dst("component"))
    .aggMsgs(F.min(Pregel.msg()))
    .run()
)

num_components = cc_results.select("component").distinct().count()
print(f"Number of connected components: {num_components:,}")

print("Component size distribution:")
(
    cc_results.groupBy("component").count()
    .orderBy(F.desc("count"))
    .withColumn("count", F.format_number(F.col("count"), 0))
    .show(10)
)

cc_results.unpersist()

## Example 5: Shortest Paths with Pregel

Single-source shortest paths via BFS-style message passing. The source starts at distance 0; each superstep extends the frontier by one hop.

In [ ]:
# Pick the most-viewed question as source
popular_question = (
    nodes_df.filter(F.col("Type") == "Question")
    .orderBy(F.desc("ViewCount"))
    .select("id", "Title")
    .first()
)
source_id = popular_question["id"]
print(f"Source: {popular_question['Title']}")

sp_vertices = g.vertices.select("id")
sp_graph = GraphFrame(sp_vertices, g.edges.select("src", "dst"))
INF = 999999

sp_results = (
    sp_graph.pregel.setMaxIter(10)
    .setEarlyStopping(True)
    .withVertexColumn(
        "distance",
        F.when(F.col("id") == source_id, F.lit(0)).otherwise(F.lit(INF)),
        F.least(F.col("distance"), F.coalesce(Pregel.msg(), F.lit(INF))),
    )
    .sendMsgToDst(F.when(Pregel.src("distance") < F.lit(INF), Pregel.src("distance") + F.lit(1)))
    .sendMsgToSrc(F.when(Pregel.dst("distance") < F.lit(INF), Pregel.dst("distance") + F.lit(1)))
    .aggMsgs(F.min(Pregel.msg()))
    .run()
)

print("Distance distribution from source:")
sp_results.filter(F.col("distance") < INF).groupBy("distance").count().orderBy("distance").show(20)

reachable = sp_results.filter(F.col("distance") < INF).count()
total = sp_results.count()
print(f"Reachable: {reachable:,} / {total:,} ({reachable/total:.1%})")

sp_results.unpersist()

## Example 6: Reputation Propagation with Pregel

A **custom algorithm** — propagate user reputation through the answer graph to compute an "authority" score for each question. Reputation flows: User → Answer → Question over 2 Pregel supersteps.

In [ ]:
# Build reputation subgraph
user_nodes = nodes_df.filter(F.col("Type") == "User").select(
    "id", F.col("Reputation").cast("double").alias("reputation"), F.lit("User").alias("Type")
)
answer_nodes = nodes_df.filter(F.col("Type") == "Answer").select(
    "id", F.col("Score").cast("double").alias("score"), F.lit("Answer").alias("Type")
)
question_nodes = nodes_df.filter(F.col("Type") == "Question").select(
    "id", F.col("ViewCount").cast("double").alias("views"), F.lit("Question").alias("Type")
)

rep_vertices = (
    user_nodes.withColumn("score", F.lit(0.0)).withColumn("views", F.lit(0.0))
    .unionByName(answer_nodes.withColumn("reputation", F.lit(0.0)).withColumn("views", F.lit(0.0)))
    .unionByName(question_nodes.withColumn("reputation", F.lit(0.0)).withColumn("score", F.lit(0.0)))
    .na.fill(0.0)
)

posts_edges = edges_df.filter(F.col("relationship") == "Posts").select("src", "dst")
answers_edges = edges_df.filter(F.col("relationship") == "Answers").select("src", "dst")
rep_edges = posts_edges.unionByName(answers_edges)

rep_graph = GraphFrame(rep_vertices, rep_edges)
print(f"Reputation subgraph: {rep_vertices.count():,} vertices, {rep_edges.count():,} edges")

In [ ]:
# Propagate reputation through the graph
rep_results = (
    rep_graph.pregel.setMaxIter(2)
    .withVertexColumn(
        "authority",
        F.col("reputation"),
        F.coalesce(Pregel.msg(), F.lit(0.0)) + F.col("authority"),
    )
    .sendMsgToDst(
        F.when(Pregel.src("authority") > F.lit(0), Pregel.src("authority"))
    )
    .aggMsgs(F.sum(Pregel.msg()))
    .run()
)

# Top questions by authority
top_questions = (
    rep_results.filter(F.col("Type") == "Question")
    .select(F.col("id"), F.col("authority"))
    .join(
        nodes_df.filter(F.col("Type") == "Question").select("id", "Title", "ViewCount"),
        on="id",
    )
    .orderBy(F.desc("authority"))
)

print("Top 10 Questions by propagated authority:")
top_questions.show(10, truncate=60)

rep_results.unpersist()

## Example 7: Debug Trace — Message Path Tracking

A purely educational example: track message paths through a small test graph to make Pregel's mechanics visible.

In [ ]:
# Small test graph
test_vertices = spark.createDataFrame(
    [("A", "Alice"), ("B", "Bob"), ("C", "Charlie"), ("D", "David")],
    ["id", "name"],
)
test_edges = spark.createDataFrame(
    [("A", "B"), ("A", "C"), ("B", "C"), ("C", "D")],
    ["src", "dst"],
)
test_graph = GraphFrame(test_vertices, test_edges)

trace_results = (
    test_graph.pregel.setMaxIter(3)
    .withVertexColumn(
        "trace",
        F.col("id"),
        F.concat_ws(" <- ", F.coalesce(Pregel.msg(), F.lit("")), F.col("id")),
    )
    .sendMsgToDst(Pregel.src("trace"))
    .aggMsgs(F.concat_ws(" | ", F.collect_list(Pregel.msg())))
    .run()
)

print("Vertex traces after 3 iterations:")
trace_results.select("id", "name", "trace").orderBy("id").show(truncate=False)
trace_results.unpersist()

## Conclusion

We built 7 progressively complex graph algorithms:
1. **In-Degree (AggregateMessages)** — single-pass baseline
2. **In-Degree (Pregel)** — same algorithm, Pregel API
3. **PageRank** — multi-iteration convergence
4. **Connected Components** — bidirectional label propagation
5. **Shortest Paths** — BFS with early stopping
6. **Reputation Propagation** — custom multi-hop algorithm
7. **Debug Trace** — message path visualization

The core Pregel pattern:
```python
result = graph.pregel \
    .setMaxIter(n) \
    .withVertexColumn("state", initial_value, update_function) \
    .sendMsgToDst(message_expression) \
    .aggMsgs(aggregation_function) \
    .run()
```

In [ ]:
spark.stop()